# NB1: Baseline XGBoost — Raw Features & Temporal Split

This notebook sets up the problem from scratch.
We load the GRACE satellite data, explore the density observations,
and train a first XGBoost model using raw input features and a simple temporal split.

By the end you will have computed train, validation, and test scores.
These scores carry forward into the next notebooks.

In [ ]:
# ============================================================
# Colab setup — run this cell FIRST. Does nothing when run locally.
# ============================================================
import os, sys, shutil

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # 1) Clone repo (or pull if already cloned)
    if not os.path.exists("/content/ML_TND"):
        os.system("git clone -q https://github.com/lotteat11/ML_TND /content/ML_TND")
    else:
        os.system("cd /content/ML_TND && git pull -q")

    # 2) Install Python deps
    os.system("pip install -q xgboost scikit-learn pyarrow joblib")

    # 3) Download parquet from GitHub Releases (wget follows redirects reliably)
    dest = "/content/ML_TND/grace_workshop_small.parquet"
    MIN_SIZE = 200 * 1024 * 1024  # 200 MB — valid file is ~257 MB
    if os.path.exists(dest) and os.path.getsize(dest) < MIN_SIZE:
        print(f"Parquet incomplete ({os.path.getsize(dest)//1024//1024} MB) — re-downloading...")
        os.remove(dest)
    if not os.path.exists(dest):
        print("Downloading grace_workshop_small.parquet (~257 MB)...")
        ret = os.system(
            f"wget -q --show-progress -O {dest} "
            "https://github.com/lotteat11/ML_TND/releases/download/v1.0-workshop/grace_workshop_small.parquet"
        )
        if ret != 0 or not os.path.exists(dest) or os.path.getsize(dest) < MIN_SIZE:
            if os.path.exists(dest): os.remove(dest)
            raise RuntimeError("Download failed or incomplete. Check your internet connection and re-run this cell.")
        print("Download complete.")

    # 4) chdir into workshop/ so all relative paths resolve
    os.chdir("/content/ML_TND/workshop")
    print("Setup complete — ready to run.")
else:
    print("Local run — no setup needed.")


In [ ]:
# Install required packages — safe to run even if already installed
%pip install -q xgboost scikit-learn pandas numpy matplotlib scipy joblib pyarrow

## About the workshop dataset

`grace_workshop.parquet` is a filtered version of the full GRACE mission data (2009–2016).  
The full dataset has 40 million rows. For the workshop we use 7 million rows chosen to keep  
the most important variation while staying fast to load and train.

**Five periods are included:**

| Period | Role | Mean altitude |
|---|---|---|
| Jan–Mar 2009 | Edge year — quiet thermosphere | ~476 km |
| Apr–Jun 2010 | Core training | ~474 km |
| Apr–Jun 2012 | Core training | ~461 km |
| Apr–Jun 2014 | Core training | ~432 km |
| Jan–Mar 2016 | Edge year — storm activity (18 Feb storm) | ~381 km |

The two edge periods are used only for out-of-sample evaluation — not for training.

Load the Python libraries needed for data handling, plotting, and machine learning.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import MinMaxScaler

from paths import GRACE_WORKSHOP as GRACE_MERGED

### Load the data and define the target

**The idea:** NRLMSISE-2.1 (MSIS) is a physics-based empirical model that already does a reasonable job estimating thermospheric density from solar and orbital inputs. Instead of predicting density from scratch, we let MSIS do the heavy lifting and train the ML model to predict only *what MSIS gets wrong*.

**Step by step:**

1. MSIS gives us a baseline prediction: **ρ_MSIS**
2. The satellite accelerometer gives us the true observed density: **ρ_obs**
3. The ratio ρ_obs / ρ_MSIS tells us how far off MSIS is — a value of 1.2 means MSIS underestimates by 20%
4. We take the **natural log** of that ratio:

$$\log_{\text{ratio}} = \log\left(\frac{\rho_{\text{obs}}}{\rho_{\text{MSIS}}}\right)$$

**Why log space?**  
- Density spans orders of magnitude (∼10⁻¹⁴ to 10⁻¹¹ kg m⁻³) — the log compresses this to a manageable range centred near zero  
- A log-ratio of 0 means MSIS is perfect; positive means MSIS underestimates; negative means MSIS overestimates  
- It reflects the log-normal distribution of density and avoids unphysical negative predictions  

**Getting back to density** — the model predicts `log_ratio`, but we need ρ. We reverse the log step:

Start from the definition:
$$\log_{\text{ratio}} = \log\left(\frac{\rho_{\text{obs}}}{\rho_{\text{MSIS}}}\right)$$

Take exp on both sides:
$$e^{\log_{\text{ratio}}} = \frac{\rho_{\text{obs}}}{\rho_{\text{MSIS}}}$$

Multiply both sides by ρ_MSIS:
$$\rho_{\text{pred}} = \rho_{\text{MSIS}} \times e^{\log_{\text{ratio}}}$$

So the model never replaces MSIS — it *corrects* it. When the model predicts log_ratio = 0, it agrees with MSIS. When it predicts log_ratio > 0, it says "MSIS is too low here".


In [ ]:
df = pd.read_parquet(GRACE_MERGED)
df['time'] = pd.to_datetime(df['grace_time'])
df = df[(df['time'] > '2009-06-01') & (df['time'] < '2016-01-01')].sort_values('time').reset_index(drop=True)
df['log_ratio'] = np.log(df['rho_obs'] / df['msis_rho'])

print(f'Rows: {len(df):,}')
print(f'Date range: {df["time"].min().date()} → {df["time"].max().date()}')
df.head(3)

## 1. Data  
Before deciding how to model it, it helps to look at the raw numbers.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(df['time'][::50], df['rho_obs'][::50], lw=0.3, color='steelblue')
ax.set_ylabel('ρ_obs [kg m⁻³]')
ax.set_title('Observed thermospheric density — full mission (every 50th point)')
plt.tight_layout()
plt.show()


## 2. MSIS already explains most of the magnitude

The empirical model NRLMSISE-2.1 (MSIS) gives us a physics-based first guess at the density.  
Instead of predicting raw density directly, the model learns the residual: `log(ρ_obs / ρ_MSIS)`.

In [ ]:
week_2010 = df[(df['time'] >= '2010-05-01') & (df['time'] < '2010-05-08')]
week_2014 = df[(df['time'] >= '2014-05-01') & (df['time'] < '2014-05-08')]

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharey='row')

for col, (week, year) in enumerate([(week_2010, '2010'), (week_2014, '2014')]):
    axes[0, col].plot(week['time'], week['rho_obs'],  lw=0.6, label='ρ_obs',  color='black')
    axes[0, col].plot(week['time'], week['msis_rho'], lw=0.6, label='ρ_MSIS', color='C1', alpha=0.7)
    axes[0, col].set_ylabel('ρ [kg m⁻³]')
    axes[0, col].legend(loc='upper right', fontsize=8)
    axes[0, col].set_title(f'{year}  —  mean alt {week["alt_km"].mean():.0f} km')
    axes[0, col].tick_params(axis='x', rotation=30)

    axes[1, col].plot(week['time'], np.log(week['rho_obs']),  lw=0.6, label='log(ρ_obs)',  color='black')
    axes[1, col].plot(week['time'], np.log(week['msis_rho']), lw=0.6, label='log(ρ_MSIS)', color='C1', alpha=0.7)
    axes[1, col].set_ylabel('log density')
    axes[1, col].legend(loc='upper right', fontsize=8)
    axes[1, col].tick_params(axis='x', rotation=30)

    axes[2, col].plot(week['time'], week['log_ratio'], lw=0.6, color='C2')
    axes[2, col].axhline(0, color='gray', lw=0.8, linestyle='--')
    axes[2, col].set_ylabel('log(ρ_obs / ρ_MSIS)')
    axes[2, col].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

The residual `log(ρ_obs / ρ_MSIS)` is what the model learns.
Look at its distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df['log_ratio'], bins=100, color='steelblue', edgecolor='none')
ax.axvline(0, color='C3', lw=1.2, linestyle='--', label='zero (MSIS = obs)')
ax.set_xlabel('log(ρ_obs / ρ_MSIS)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Raw input features

These are the signals available as inputs — no transforms applied yet.

In [ ]:
feature_cols = ['f107', 'f107a', 'ap_m3h', 'ap_m6h', 'alt_km', 'lat', 'lon', 'matched_tec_value']

df[feature_cols].describe().round(3)

Plot each raw input feature over time.

In [ ]:
week_2010 = df[(df['time'] >= '2010-05-01') & (df['time'] < '2010-05-08')]
week_2014 = df[(df['time'] >= '2014-05-01') & (df['time'] < '2014-05-08')]

feature_cols = ['f107', 'f107a', 'ap_m3h', 'ap_m6h', 'alt_km', 'lat', 'lon', 'matched_tec_value']
all_cols = ['rho_obs'] + feature_cols
colors   = ['black', 'C0', 'C1', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8']
n_rows   = len(all_cols)

fig, axes = plt.subplots(n_rows, 2, figsize=(14, 2.2 * n_rows), sharex='col')

for col_idx, (week, year) in enumerate([(week_2010, '2010'), (week_2014, '2014')]):
    for row_idx, (var, c) in enumerate(zip(all_cols, colors)):
        ax = axes[row_idx, col_idx]
        ax.plot(week['time'], week[var], lw=0.4, color=c)
        ax.set_ylabel(var, fontsize=8)
        if row_idx == 0:
            ax.set_title(f'{year}  —  mean alt {week["alt_km"].mean():.0f} km')
        if row_idx == n_rows - 1:
            ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 4. XGBoost — the model

**Gradient boosted trees** build an ensemble of decision trees sequentially.  
Each new tree learns to correct the mistakes of all the trees before it.

Key hyperparameters:
- `MAX_DEPTH` — how deep each tree can grow. A deeper tree can capture more complex patterns,  
  but a tree that is too deep will memorise the training data rather than generalise.
- `N_ESTIMATORS` — how many trees to build. More trees means more capacity,  
  but at some point adding more trees stops helping.
- `LEARNING_RATE` — how much weight each new tree gets. Lower values make the model more conservative.

The **train / val / test split** divides the data in time: train comes first, then val, then test.  
The model never sees val or test data during training — these measure how well it generalises.

Adjust the parameters below, then rerun the training cells.

In [ ]:
# ── PARAMETERS ──────────────────────────────────────────────────────────────
RAW_FEATURES = ["f107", "f107a", "ap_m3h", "ap_m6h", "alt_km", "lat", "lon", "matched_tec_value"]
TARGET       = "log_ratio"

TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
# TEST_FRAC  = 1 - TRAIN_FRAC - VAL_FRAC

MAX_DEPTH     = 4      # try 2, 4, 6, 8 — watch the gap open with higher values
N_ESTIMATORS  = 500
LEARNING_RATE = 0.05
# ────────────────────────────────────────────────────────────────────────────

Divide the data into three consecutive time blocks: train, validation, and test.

In [ ]:
# Drop rows with missing values in any feature or target
df = df.dropna(subset=RAW_FEATURES + [TARGET]).reset_index(drop=True)

n = len(df)
n_train = int(n * TRAIN_FRAC)
n_val   = int(n * (TRAIN_FRAC + VAL_FRAC))

df_train = df.iloc[:n_train]
df_val   = df.iloc[n_train:n_val]
df_test  = df.iloc[n_val:]

print(f"Train: {len(df_train):>7,}  {df_train['time'].min().date()} → {df_train['time'].max().date()}")
print(f"Val:   {len(df_val):>7,}  {df_val['time'].min().date()} → {df_val['time'].max().date()}")
print(f"Test:  {len(df_test):>7,}  {df_test['time'].min().date()} → {df_test['time'].max().date()}")
print(f"NaN check — any NaN in features: {df[RAW_FEATURES].isna().any().any()}")

Scale all features to the same numerical range (required for numerical stability), then train XGBoost.

Scale features to the range (−1, 1) and train XGBoost on the training set.

In [ ]:
# Scale features (X) — fit on training data only, apply to all splits
scaler_X = MinMaxScaler(feature_range=(-1, 1))
X_train = scaler_X.fit_transform(df_train[RAW_FEATURES])
X_val   = scaler_X.transform(df_val[RAW_FEATURES])
X_test  = scaler_X.transform(df_test[RAW_FEATURES])

# Scale target (y) — fit on training data only, apply to all splits
scaler_y = MinMaxScaler(feature_range=(-1, 1))
y_train = scaler_y.fit_transform(df_train[[TARGET]]).ravel()
y_val   = scaler_y.transform(df_val[[TARGET]]).ravel()
y_test  = scaler_y.transform(df_test[[TARGET]]).ravel()

model = xgb.XGBRegressor(
    max_depth=MAX_DEPTH,
    n_estimators=N_ESTIMATORS,
    learning_rate=LEARNING_RATE,
    subsample=0.5,
    colsample_bytree=0.6,
    min_child_weight=300,
    tree_method='hist',
    n_jobs=-1,
    early_stopping_rounds=20,
    verbosity=0,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print(f'Best iteration: {model.best_iteration}')

Measure how far predictions deviate from the true values on each of the three sets, then plot as a bar chart.

In [ ]:
# Convert predictions back to physical density space
# rho_pred = rho_msis * exp(predicted log_ratio)
rho_obs_train  = df_train['rho_obs'].values
rho_pred_train = df_train['msis_rho'].values * np.exp(scaler_y.inverse_transform(model.predict(X_train).reshape(-1,1)).ravel())
rho_msis_train = df_train['msis_rho'].values

rho_obs_val    = df_val['rho_obs'].values
rho_pred_val   = df_val['msis_rho'].values  * np.exp(scaler_y.inverse_transform(model.predict(X_val).reshape(-1,1)).ravel())

rho_obs_test   = df_test['rho_obs'].values
rho_pred_test  = df_test['msis_rho'].values * np.exp(scaler_y.inverse_transform(model.predict(X_test).reshape(-1,1)).ravel())

rmse_train = root_mean_squared_error(rho_obs_train, rho_pred_train)
rmse_val   = root_mean_squared_error(rho_obs_val,   rho_pred_val)
rmse_test  = root_mean_squared_error(rho_obs_test,  rho_pred_test)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Train', 'Val', 'Test'], [rmse_train, rmse_val, rmse_test],
              color=['steelblue', 'orange', 'green'], alpha=0.85, width=0.5)
ax.bar_label(bars, fmt='{:.2e}', padding=4)
ax.set_ylabel('RMSE  [kg m⁻³]')
ax.set_title(f'NB1 — Train/Val/Test gap  (MAX_DEPTH={MAX_DEPTH})')
ax.set_ylim(0, max(rmse_train, rmse_val, rmse_test) * 1.25)
plt.tight_layout()
plt.show()

print(f'Train {rmse_train:.3e}  |  Val {rmse_val:.3e}  |  Test {rmse_test:.3e}')

Plot the model's density predictions alongside the satellite observations and the MSIS baseline.

In [ ]:
pred_log_val = scaler_y.inverse_transform(model.predict(X_val).reshape(-1, 1)).ravel()
rho_pred_val = df_val['msis_rho'].values * np.exp(pred_log_val)

# Validation — full validation period (every 10th point)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_val['time'].values[::10], df_val['rho_obs'].values[::10],  lw=0.5, label='Observed',  color='black')
ax.plot(df_val['time'].values[::10], df_val['msis_rho'].values[::10], lw=0.5, label='MSIS',      color='C1', alpha=0.8)
ax.plot(df_val['time'].values[::10], rho_pred_val[::10], lw=0.5, label='Predicted', color='C0', alpha=0.9)
ax.set_yscale('log')
ax.set_ylabel('ρ [kg m⁻³]')
ax.set_xlabel('Time')
ax.set_title('Validation — full period')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


The full-period plot shows long-term behaviour. Zooming into **3 days** reveals whether the model captures the ~90-minute orbital oscillation in density that MSIS already models well, and where the residual improvement comes from.

In [ ]:
# Validation — 3-day zoom
t_end3 = pd.Timestamp(df_val['time'].values[0]) + pd.Timedelta(days=3)
m3 = df_val['time'].values < t_end3
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_val['time'].values[m3], df_val['rho_obs'].values[m3],  lw=0.8, label='Observed',  color='black')
ax.plot(df_val['time'].values[m3], df_val['msis_rho'].values[m3], lw=0.8, label='MSIS',      color='C1', alpha=0.8)
ax.plot(df_val['time'].values[m3], rho_pred_val[m3], lw=0.8, label='Predicted', color='C0', alpha=0.9)
ax.set_yscale('log')
ax.set_ylabel('ρ [kg m⁻³]')
ax.set_xlabel('Time')
ax.set_title('Validation — 3-day zoom')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


Plot predicted vs actual values as a scatter. A perfect model would follow the dashed diagonal line.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (df_split, X_split, label) in zip(axes, [
    (df_val,  X_val,  'Val'),
    (df_test, X_test, 'Test'),
]):
    rho_obs  = df_split['rho_obs'].values
    rho_msis = df_split['msis_rho'].values
    rho_pred = rho_msis * np.exp(scaler_y.inverse_transform(model.predict(X_split).reshape(-1,1)).ravel())

    lims = [min(rho_obs.min(), rho_msis.min(), rho_pred.min()),
            max(rho_obs.max(), rho_msis.max(), rho_pred.max())]

    ax.scatter(rho_obs, rho_msis, s=1, alpha=0.1, color='C1', rasterized=True, label='MSIS')
    ax.scatter(rho_obs, rho_pred, s=1, alpha=0.1, color='C0', rasterized=True, label='Model')
    ax.plot(lims, lims, 'k--', lw=1)
    ax.set_xlabel('Observed ρ [kg m⁻³]')
    ax.set_ylabel('Modelled ρ [kg m⁻³]')
    ax.set_title(label)
    ax.legend(markerscale=8, fontsize=9)

plt.suptitle('Parity — MSIS vs model against observations')
plt.tight_layout()
plt.show()

Compute RMSE, MAPE, and R² in physical density space for both MSIS and the model prediction.

In [ ]:
from sklearn.metrics import r2_score

def density_metrics(rho_obs, rho_pred, label):
    mask = rho_obs > 0
    obs, pred = rho_obs[mask], rho_pred[mask]
    rmse = np.sqrt(np.mean((obs - pred) ** 2))
    mape = np.mean(np.abs((obs - pred) / obs)) * 100
    r2   = r2_score(obs, pred)
    return {"Model": label, "RMSE [kg/m³]": f"{rmse:.3e}",
            "MAPE [%]": f"{mape:.1f}", "R²": f"{r2:.3f}"}

# Convert log-ratio predictions back to density space
# rho_pred = rho_msis * exp(predicted log_ratio)
rho_pred_val_full  = df_val["msis_rho"].values  * np.exp(scaler_y.inverse_transform(model.predict(X_val).reshape(-1,1)).ravel())
rho_pred_test_full = df_test["msis_rho"].values * np.exp(scaler_y.inverse_transform(model.predict(X_test).reshape(-1,1)).ravel())

rows = [
    density_metrics(df_val["rho_obs"].values,  df_val["msis_rho"].values,  "MSIS — Val"),
    density_metrics(df_val["rho_obs"].values,  rho_pred_val_full,           "Model — Val"),
    density_metrics(df_test["rho_obs"].values, df_test["msis_rho"].values,  "MSIS — Test"),
    density_metrics(df_test["rho_obs"].values, rho_pred_test_full,          "Model — Test"),
]

import pandas as pd
pd.DataFrame(rows).set_index("Model")

## Try this

Is this a good model? Look at the bar chart and the parity plot.

**Hyperparameters** — change one, rerun from the Parameters cell:
- `MAX_DEPTH = 2` then `8` — what happens to the train vs val gap?
- `N_ESTIMATORS = 50` — at what point do more trees stop helping?

**Inputs** — edit `RAW_FEATURES` and retrain:
- Remove `"matched_tec_value"` — how much does ionospheric information contribute?
- Keep only `["f107", "alt_km"]` — what is the floor?

**Split** — try `TRAIN_FRAC = 0.50` then `0.90`. Does more training data help?

In **NB2** we keep the same split and model but change the features.